# Path E — published-MIC lookup for the 6 Syn3A drug-target enzymes

Companion to `memory_bank/facts/measured/toxicity_path_e_saturating_inhibitor_v0.json`. For each of the six Syn3A SBML drug-target enzymes (ileRS / leuRS / metRS / asnRS / tmk / trxB), fetches:

1. The UniProt ortholog in *M. mycoides* SC (the parent organism) and *M. pneumoniae* / *M. genitalium* if available — gives a stable target accession to query downstream.
2. ChEMBL target ID for that ortholog, then ChEMBL bioactivity records for the canonical drug (mupirocin / tavaborole / REP3123 / etc.) — looking specifically for organism = Mycoplasma in the assay metadata.
3. PubChem BioAssay search for any Mycoplasma-organism activity records on the canonical compounds (CID lookup → AID list → activity values).

All four endpoints (UniProt / ChEMBL / PubChem / NCBI eutils) are 403 from the Claude Code sandbox. They work from Colab. Wall estimate: ~5-10 minutes (mostly API rate limiting). No GPU needed.

Output: `outputs/toxicity/path_e_mic_lookup.json` consolidating all hits per (target, compound). Auto-pushed at the end.

In [ ]:
# Cell 1 — install + clone + PAT prompt.
!pip install -q requests>=2.31
import os, subprocess, getpass
BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if r.returncode != 0: raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
%cd /content/cell

if not os.environ.get("GITHUB_PAT", "").strip():
    pat = getpass.getpass("Paste your GitHub PAT (input hidden): ").strip()
    if not pat: raise ValueError("empty PAT")
    os.environ["GITHUB_PAT"] = pat
print(f"PAT set ({len(os.environ['GITHUB_PAT'])} chars)")

In [ ]:
# Cell 2 — define the 6 targets + canonical drugs from the Path E fact.
TARGETS = [
    {"locus": "JCVISYN3A_0519", "gene": "ileRS",
     "enzyme": "isoleucyl-tRNA synthetase",
     "drug_name": "Mupirocin", "drug_synonyms": ["pseudomonic acid A"],
     "uniprot_query": "isoleucyl-tRNA AND (taxonomy_id:272632 OR taxonomy_id:272634 OR taxonomy_id:243273)"},
    {"locus": "JCVISYN3A_0634", "gene": "leuRS",
     "enzyme": "leucyl-tRNA synthetase",
     "drug_name": "Tavaborole", "drug_synonyms": ["AN2690", "GSK2251052", "AN3365"],
     "uniprot_query": "leucyl-tRNA AND (taxonomy_id:272632 OR taxonomy_id:272634 OR taxonomy_id:243273)"},
    {"locus": "JCVISYN3A_0012", "gene": "metRS",
     "enzyme": "methionyl-tRNA synthetase",
     "drug_name": "REP3123", "drug_synonyms": ["REP-3123"],
     "uniprot_query": "methionyl-tRNA AND (taxonomy_id:272632 OR taxonomy_id:272634 OR taxonomy_id:243273)"},
    {"locus": "JCVISYN3A_0045", "gene": "tmk",
     "enzyme": "dTMP kinase",
     "drug_name": "5-bromo-2'-deoxyuridine 5'-monophosphate", "drug_synonyms": ["BrdUMP", "5-BrdUMP"],
     "uniprot_query": "dTMP kinase AND (taxonomy_id:272632 OR taxonomy_id:272634 OR taxonomy_id:243273)"},
    {"locus": "JCVISYN3A_0076", "gene": "asnRS",
     "enzyme": "asparaginyl-tRNA synthetase",
     "drug_name": "Tirandamycin B", "drug_synonyms": ["tirandamycin"],
     "uniprot_query": "asparaginyl-tRNA AND (taxonomy_id:272632 OR taxonomy_id:272634 OR taxonomy_id:243273)"},
    {"locus": "JCVISYN3A_0819", "gene": "trxB",
     "enzyme": "thioredoxin-disulfide reductase",
     "drug_name": "Auranofin", "drug_synonyms": ["SK&F-39162"],
     "uniprot_query": "thioredoxin-disulfide reductase AND (taxonomy_id:272632 OR taxonomy_id:272634 OR taxonomy_id:243273)"},
]
for t in TARGETS:
    print(f"  {t['locus']}  {t['gene']:6s}  drug={t['drug_name']}")

In [ ]:
# Cell 3 — Layer 1: UniProt orthology lookup for each target's enzyme
# in M. mycoides SC / M. pneumoniae / M. genitalium.
import requests, time, json

UA = {"User-Agent": "cell-sim-bot/1.0"}

def uniprot_search(query, fields="accession,protein_name,organism_id,length,xref_pfam,xref_chembl"):
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": query, "format": "tsv", "fields": fields, "size": "5"},
                     headers=UA, timeout=30)
    r.raise_for_status()
    return r.text

L1 = {}
for t in TARGETS:
    print(f"\n=== {t['gene']} ({t['enzyme']}) ===")
    try:
        body = uniprot_search(t["uniprot_query"])
    except Exception as e:
        body = f"FAIL: {type(e).__name__}: {e}"
    print(body[:800])
    L1[t["locus"]] = body
    time.sleep(0.5)

In [ ]:
# Cell 4 — Layer 2: ChEMBL compound + activity lookup for each canonical drug.
# Strategy: search compounds by drug name -> get ChEMBL ID -> fetch activity
# data filtered by organism contains 'Mycoplasma'.
import requests, json, time
CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_compound_search(name):
    r = requests.get(f"{CHEMBL}/molecule.json",
                     params={"molecule_synonyms__synonyms__icontains": name, "limit": "5"},
                     headers={**UA, "Accept": "application/json"}, timeout=30)
    if r.status_code != 200:
        return None
    return r.json()

def chembl_activities_by_chembl_id(chembl_id):
    out = []
    offset = 0
    while True:
        r = requests.get(f"{CHEMBL}/activity.json",
                         params={"molecule_chembl_id": chembl_id,
                                 "assay_organism__icontains": "Mycoplasma",
                                 "limit": "500", "offset": str(offset)},
                         headers={**UA, "Accept": "application/json"}, timeout=60)
        if r.status_code != 200:
            break
        data = r.json()
        acts = data.get("activities", [])
        out.extend(acts)
        if not data.get("page_meta", {}).get("next"):
            break
        offset += 500
        time.sleep(0.3)
    return out

L2 = {}
for t in TARGETS:
    print(f"\n=== ChEMBL: {t['drug_name']} (synonyms {t['drug_synonyms']}) ===")
    summary = {"compound_search": None, "chembl_id": None, "mycoplasma_activities": []}
    try:
        for name in [t["drug_name"]] + t["drug_synonyms"]:
            res = chembl_compound_search(name)
            if res and res.get("molecules"):
                cid = res["molecules"][0].get("molecule_chembl_id")
                if cid:
                    summary["compound_search"] = name
                    summary["chembl_id"] = cid
                    print(f"  resolved {name!r} -> {cid}")
                    break
        if summary["chembl_id"]:
            acts = chembl_activities_by_chembl_id(summary["chembl_id"])
            for a in acts:
                summary["mycoplasma_activities"].append({
                    "organism": a.get("assay_organism"),
                    "standard_type": a.get("standard_type"),
                    "standard_value": a.get("standard_value"),
                    "standard_units": a.get("standard_units"),
                    "target_pref_name": a.get("target_pref_name"),
                    "document_chembl_id": a.get("document_chembl_id"),
                })
            print(f"  {len(acts)} Mycoplasma activity records")
            for a in summary["mycoplasma_activities"][:5]:
                print(f"    {a['organism']:30s} {a['standard_type']:10s} "
                      f"{a['standard_value']} {a['standard_units']}")
        else:
            print(f"  no compound resolved")
    except Exception as e:
        summary["error"] = f"{type(e).__name__}: {e}"
        print(f"  FAIL: {e}")
    L2[t["locus"]] = summary
    time.sleep(0.5)

In [ ]:
# Cell 5 — Layer 3: PubChem BioAssay search for each compound.
# Looks for assay records keyed to the compound's CID where the assay
# title contains 'Mycoplasma'. PubChem's REST API is much more
# permissive than ChEMBL's organism filter so this often catches
# records ChEMBL misses.
import requests, time, json
PUG = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

def pubchem_cid_for_name(name):
    r = requests.get(f"{PUG}/compound/name/{requests.utils.quote(name)}/cids/JSON",
                     headers=UA, timeout=30)
    if r.status_code != 200:
        return None
    cids = r.json().get("IdentifierList", {}).get("CID", [])
    return cids[0] if cids else None

def pubchem_assays_for_cid(cid):
    # Get assay summary
    r = requests.get(f"{PUG}/compound/cid/{cid}/assaysummary/JSON",
                     headers=UA, timeout=60)
    if r.status_code != 200:
        return []
    rows = r.json().get("Table", {}).get("Row", [])
    cols = r.json().get("Table", {}).get("Columns", {}).get("Column", [])
    return rows, cols

L3 = {}
for t in TARGETS:
    print(f"\n=== PubChem: {t['drug_name']} ===")
    summary = {"cid": None, "mycoplasma_hits": []}
    try:
        for name in [t["drug_name"]] + t["drug_synonyms"]:
            cid = pubchem_cid_for_name(name)
            if cid:
                summary["cid"] = cid
                print(f"  resolved {name!r} -> CID {cid}")
                break
        if summary["cid"]:
            rows, cols = pubchem_assays_for_cid(summary["cid"])
            col_names = [c.get("@Name") if isinstance(c, dict) else str(c) for c in cols] if cols else []
            for row in rows or []:
                cells = row.get("Cell", [])
                row_str = " | ".join(str(c) for c in cells)
                if "mycoplasma" in row_str.lower():
                    summary["mycoplasma_hits"].append(row_str[:300])
            print(f"  {len(summary['mycoplasma_hits'])} mycoplasma-bearing assay records")
            for h in summary["mycoplasma_hits"][:5]:
                print(f"    {h}")
        else:
            print(f"  no CID resolved")
    except Exception as e:
        summary["error"] = f"{type(e).__name__}: {e}"
        print(f"  FAIL: {e}")
    L3[t["locus"]] = summary
    time.sleep(0.5)

In [ ]:
# Cell 6 — consolidate + auto-push.
import json, os, subprocess
from pathlib import Path

out = {
    "targets": TARGETS,
    "layer_1_uniprot_orthology": L1,
    "layer_2_chembl_mycoplasma_activities": L2,
    "layer_3_pubchem_mycoplasma_assays": L3,
}
out_path = Path("outputs/toxicity/path_e_mic_lookup.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(out, indent=2, default=str))
print(f"wrote {out_path} ({out_path.stat().st_size} bytes)")

pat = os.environ.get("GITHUB_PAT", "").strip()
if not pat: raise SystemExit("GITHUB_PAT not set")
_run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
_run(["git", "config", "user.name", "cell-sim-bot"])
_run(["git", "add", "-f", str(out_path)])
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
if status.stdout.strip():
    _run(["git", "commit", "-m", "Session 27: Path E published-MIC lookup for 6 drug-target enzymes"])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    _run(["git", "push", remote, BRANCH])
    print("\npush complete.")
else:
    print("nothing changed")